# 06b — E1 leave-one-generator-out for M4 (Akshat Yuvan)

The E1 protocol of `06-e1-logo`, applied to M4 (AST), which did not exist
when that sweep ran. Same three folds (MusicGen_medium, stable_audio_open,
mustango), same row selection, same `pos_weight` class weighting, same
closed-set and open-set test sets, same metric code. M4 is trained exactly as
in `09-m4-ast` (pre-trained AST, Adam 2e-5, warmup + cosine, 4 epochs, fp16,
best plain val accuracy). Each fold writes `e1_{fold}_M4.json` the moment it
finishes; folds already present in `fmc-e1-results` are skipped.

In [ ]:
import os, glob, json, time, math, copy
import numpy as np, pandas as pd, torch, torch.nn as nn
import transformers
from transformers import ASTFeatureExtractor, ASTForAudioClassification

print(torch.__version__, transformers.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available()

def find_input(name, marker):
    d = f'/kaggle/input/datasets/akshatyuvan/{name}'
    if os.path.exists(f'{d}/{marker}'):
        return d
    hits = glob.glob(f'/kaggle/input/**/{name}/**/{marker}', recursive=True)
    assert hits, f'{name}/{marker} not found'
    return os.path.dirname(hits[0])

MELS_DIR = find_input('fmc-m2-mels', 'subset.csv')
RAW_DIR  = find_input('fakemusiccaps-raw', 'mustango')
REAL_DIR = find_input('musiccaps-real-16k', '-0Gj8-vB1q4.wav')
OUT, SEED = '/kaggle/working', 42
CKPT_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

sub    = pd.read_csv(f'{MELS_DIR}/subset.csv', dtype={'ytid': str})
labels = np.load(f'{MELS_DIR}/labels.npy')
assert (sub.label.values == labels).all() and len(sub) == 10697
# Raw-audio paths, remapped exactly as the M2 notebook did.
sub['path'] = (sub.path
    .str.replace('/kaggle/working/real', REAL_DIR, regex=False)
    .str.replace('/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw', RAW_DIR, regex=False))
assert sub.path.map(os.path.exists).all()
idx = {s: np.where((sub.split == s).values)[0] for s in ['train', 'val', 'test']}
print({s: len(v) for s, v in idx.items()})

In [ ]:
# AST features for all 10,697 clips, held in RAM as fp16 (2.8 GB).
import soundfile as sf
from joblib import Parallel, delayed

N_SAMPLES = 160000
fe = ASTFeatureExtractor.from_pretrained(CKPT_NAME)
assert (fe.sampling_rate, fe.num_mel_bins, fe.max_length) == (16000, 128, 1024)
print('AudioSet normalisation mean/std:', fe.mean, fe.std)

def ast_feat(path):
    y, sr = sf.read(path, dtype='float32')
    assert sr == 16000
    y = y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))
    return fe(y, sampling_rate=16000, return_tensors='np')['input_values'][0].astype(np.float16)

t0 = time.time()
FB = np.empty((len(sub), 1024, 128), dtype=np.float16)
for s in range(0, len(sub), 1000):
    FB[s:s+1000] = np.stack(Parallel(n_jobs=4)(delayed(ast_feat)(p) for p in sub.path.values[s:s+1000]))
print('features', FB.shape, f'{time.time()-t0:.0f}s', '| NaNs', int(np.isnan(FB[:200].astype(np.float32)).sum()))

In [ ]:
# Data, model, training loop.
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

class FbDS(Dataset):
    def __init__(self, arr, rows, ys):
        self.arr, self.rows, self.ys = arr, rows, ys
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        return (torch.from_numpy(self.arr[self.rows[i]].astype(np.float32)),
                torch.tensor(float(self.ys[i])))

def loader(arr, rows, ys, shuffle=False, gen=None, bs=8):
    return DataLoader(FbDS(arr, rows, ys), batch_size=bs, shuffle=shuffle, generator=gen,
                      num_workers=2, pin_memory=True)

def make_m4(attn='sdpa'):
    return ASTForAudioClassification.from_pretrained(
        CKPT_NAME, num_labels=1, ignore_mismatched_sizes=True, attn_implementation=attn)

def predict(model, dl):
    model.eval(); out = []
    with torch.no_grad():
        for xb, _ in dl:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out.append(torch.sigmoid(model(xb.cuda()).logits.squeeze(1).float()).cpu().numpy())
    return np.concatenate(out)

def metrics(true, score):
    """Line-for-line the M1/M2/M3 metric code."""
    pred = (score > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(true, score)
    eer = float(fpr[np.nanargmin(np.abs((1 - tpr) - fpr))])
    return {"balanced_accuracy": float(balanced_accuracy_score(true, pred)),
            "precision": float(precision_score(true, pred, zero_division=0)),
            "recall":    float(recall_score(true, pred)),
            "f1":        float(f1_score(true, pred)),
            "auc":       float(roc_auc_score(true, score)),
            "eer":       eer}

EPOCHS, PEAK_LR, CLIP = 4, 2e-5, 5.0

def train_m4(model, dl_tr, dl_va, y_va, pos_weight, ckpt):
    steps = EPOCHS * len(dl_tr); warm = len(dl_tr)
    def lr_lambda(s):
        if s < warm: return 0.1 + 0.9 * s / warm
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm)))
    opt = torch.optim.Adam(model.parameters(), lr=PEAK_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device='cuda'))
    scaler = torch.amp.GradScaler('cuda')
    best, best_ep, best_state, hist, t0 = -1.0, -1, None, [], time.time()
    for ep in range(EPOCHS):
        model.train(); run, n = 0.0, 0
        for xb, yb in dl_tr:
            xb, yb = xb.cuda(non_blocking=True), yb.cuda(non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = lossf(model(xb).logits.squeeze(1), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item() * len(yb); n += len(yb)
        s_va = predict(model, dl_va)
        va = float(((s_va > 0.5).astype(int) == y_va).mean())
        hist.append({"epoch": ep, "train_loss": round(run / n, 4), "val_acc": round(va, 4),
                     "val_bal_acc": round(float(balanced_accuracy_score(y_va, (s_va > 0.5).astype(int))), 4),
                     "elapsed_s": round(time.time() - t0, 1)})
        print(f'  epoch {ep}  train_loss {run/n:.4f}  val_acc {va:.4f}  ({time.time()-t0:.0f}s)', flush=True)
        if va > best:
            best, best_ep = va, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            torch.save({k: v.half() for k, v in best_state.items()}, ckpt)   # fp16: 172 MB
    model.load_state_dict(best_state)
    return best, best_ep, time.time() - t0, hist

In [ ]:
# Manifest for the open-set clips, and the skip-if-done logic of E1.
import shutil
MAN_DIR = find_input('fmc-manifest', 'manifest.csv')
d = '/kaggle/input/datasets/akshatyuvan/fmc-e1-results'
DONE_DIR = d if os.path.exists(f'{d}/README.md') else None
man = pd.read_csv(f'{MAN_DIR}/manifest.csv', dtype={'ytid': str})
FOLDS = ['MusicGen_medium', 'stable_audio_open', 'mustango']

def already_done(fold):
    f = f'e1_{fold}_M4.json'
    if os.path.exists(f'{OUT}/{f}'): return True
    if DONE_DIR and os.path.exists(f'{DONE_DIR}/{f}'):
        shutil.copy(f'{DONE_DIR}/{f}', f'{OUT}/{f}'); return True
    return False
print({f: already_done(f) for f in FOLDS})

def r4(dd): return {k: round(v, 4) for k, v in dd.items()}

for fold in FOLDS:
    if already_done(fold):
        print(f'== {fold}: done, skipping'); continue
    print(f'\n========== fold: withhold {fold} ==========')
    seen = sorted(set(sub.generator[sub.label == 1]) - {fold})
    keep = (sub.generator != fold).values
    tr = np.where(keep & (sub.split == 'train').values)[0]
    va = np.where(keep & (sub.split == 'val').values)[0]
    te_closed = np.where(keep & (sub.split == 'test').values)[0]
    te_real = np.where(((sub.split == 'test') & (sub.label == 0)).values)[0]
    assert len(te_real) == 804
    y_tr, y_va, y_tc = labels[tr], labels[va], labels[te_closed]
    n_real, n_syn = int((y_tr == 0).sum()), int((y_tr == 1).sum())
    pos_weight = n_real / n_syn
    print(f'train real {n_real} synth {n_syn} (pos_weight {pos_weight:.3f}) | val {len(va)} | closed {len(te_closed)}')

    # Open set: all 804 test-split clips of the withheld generator, AST
    # features from raw audio with the same function as the cached ones.
    rows = man[(man.split == 'test') & (man.generator == fold)].sort_values('ytid')
    assert len(rows) == 804 and rows.path.map(os.path.exists).all()
    OF = np.stack(Parallel(n_jobs=4)(delayed(ast_feat)(p) for p in rows.path))
    y_open = np.r_[np.zeros(len(te_real), int), np.ones(len(OF), int)]

    torch.manual_seed(SEED); np.random.seed(SEED)
    g = torch.Generator().manual_seed(SEED)
    model = make_m4().cuda()
    best, best_ep, t_s, hist = train_m4(model, loader(FB, tr, y_tr, shuffle=True, gen=g),
                                        loader(FB, va, y_va, bs=32), y_va, pos_weight,
                                        f'{OUT}/e1_{fold}_M4_fp16.pt')
    s_c = predict(model, loader(FB, te_closed, y_tc, bs=32))
    s_o = np.r_[predict(model, loader(FB, te_real, np.zeros(len(te_real)), bs=32)),
                predict(model, loader(OF, np.arange(len(OF)), np.ones(len(OF)), bs=32))]
    mc, mo = metrics(y_tc, s_c), metrics(y_open, s_o)
    res = {"experiment": "E1_leave_one_generator_out", "fold": fold, "withheld_generator": fold,
           "seen_generators": seen, "seed": SEED, "selection_metric": "val_accuracy",
           "n_train_real": n_real, "n_train_synth": n_syn, "pos_weight": round(pos_weight, 4),
           "n_val": int(len(va)), "model": "M4", "device": torch.cuda.get_device_name(0),
           "best_val_acc": round(best, 4), "best_epoch": best_ep, "train_time_s": round(t_s, 1),
           "config": f"pretrained {CKPT_NAME}, Adam {PEAK_LR} wd 1e-4, batch 8, warmup 1 epoch + cosine "
                     f"to 0.01x per step, {EPOCHS} epochs, clip {CLIP}, fp16 + GradScaler, pos_weight",
           "closed_set": {**r4(mc), "n_real": int((y_tc == 0).sum()), "n_synth": int((y_tc == 1).sum())},
           "open_set":   {**r4(mo), "n_real": int(len(te_real)), "n_synth": int(len(OF))},
           "delta_bal_acc_open_minus_closed": round(mo["balanced_accuracy"] - mc["balanced_accuracy"], 4),
           "history": hist}
    json.dump(res, open(f'{OUT}/e1_{fold}_M4.json', 'w'), indent=2)
    print(f'  M4: closed {mc["balanced_accuracy"]:.4f} -> open {mo["balanced_accuracy"]:.4f} '
          f'(withheld recall {mo["recall"]:.4f}, open AUC {mo["auc"]:.4f})', flush=True)
    del model, OF; torch.cuda.empty_cache()
print(sorted(os.listdir(OUT)))